# Compare T1 MRI intensity normalization

This notebook compares the current FreeSurfer `T1.mgz` intensities with a per-volume z-score calculated inside the corresponding FreeSurfer brain mask. It uses one deterministic training subject, displays three anatomical views, and compares the masked intensity distributions.

**Important:** `T1.mgz` is already processed by FreeSurfer, including intensity non-uniformity correction and white-matter intensity normalization. The left panel is therefore the source FreeSurfer T1 (before this notebook's z-score), not raw scanner data. The z-score is an experimental deep-learning baseline, not a clinically validated choice. This visual example alone cannot establish which preprocessing is better.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "PROJECT_CONTEXT.md").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the repository or a subdirectory."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRAIN_MANIFEST = PROJECT_ROOT / "data/processed/binary_cdr/splits/train.csv"
RANDOM_SEED = 42

if not TRAIN_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Training manifest not found: {TRAIN_MANIFEST}. "
        "Run `python -m src.data.prepare_binary_splits` from the project root first."
    )

train_manifest = pd.read_csv(TRAIN_MANIFEST)
if train_manifest.empty:
    raise ValueError("The training manifest contains no subjects.")
sample = train_manifest.sample(n=1, random_state=RANDOM_SEED).iloc[0]
source_path = PROJECT_ROOT / sample["source_image_path"]
brain_path = source_path.with_name("brainmask.mgz")

for path in (source_path, brain_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required FreeSurfer volume is missing or the external data disk is not mounted: {path}"
        )

label = "CDR = 0" if int(sample["binary_label"]) == 0 else "CDR > 0"
print(f"Training subject: {sample['participant_id']} | {label} | CDR {sample['cdr']}")
print("Source volume: FreeSurfer T1.mgz (already intensity-processed, not z-scored here)")
print(f"Brain-mask volume: {brain_path}")

In [ ]:
t1_image = nib.as_closest_canonical(nib.load(str(source_path)))
brain_image = nib.as_closest_canonical(nib.load(str(brain_path)))
if t1_image.shape != brain_image.shape or not np.allclose(t1_image.affine, brain_image.affine, atol=1e-4):
    raise ValueError(
        "T1.mgz and brainmask.mgz do not share the same voxel grid; "
        "do not apply the brain mask without explicit resampling."
    )

source_volume = np.asarray(t1_image.dataobj, dtype=np.float32)
brain_mask_image = np.asarray(brain_image.dataobj, dtype=np.float32)
brain_mask = np.isfinite(brain_mask_image) & (brain_mask_image > 0)
brain_mask &= np.isfinite(source_volume)
if not np.any(brain_mask):
    raise ValueError("The FreeSurfer brain mask contains no usable voxels.")

brain_values = source_volume[brain_mask]
mean_intensity = float(brain_values.mean())
std_intensity = float(brain_values.std(ddof=0))
if not np.isfinite(std_intensity) or std_intensity <= 0:
    raise ValueError("Cannot calculate z-score: brain-mask intensity standard deviation is zero or invalid.")

normalized_volume = np.zeros_like(source_volume, dtype=np.float32)
normalized_volume[brain_mask] = (brain_values - mean_intensity) / std_intensity
normalized_values = normalized_volume[brain_mask]

print(f"Volume shape: {source_volume.shape}")
print(f"Voxel size (mm): {tuple(round(float(v), 2) for v in t1_image.header.get_zooms()[:3])}")
print(f"Brain-mask voxels: {int(brain_mask.sum()):,}")
print(f"Source brain mean / standard deviation: {brain_values.mean():.3f} / {brain_values.std():.3f}")
print(f"Z-score brain mean / standard deviation: {normalized_values.mean():.6f} / {normalized_values.std():.6f}")

In [ ]:
brain_coordinates = np.nonzero(brain_mask)
center = tuple(int(np.median(axis_coordinates)) for axis_coordinates in brain_coordinates)
raw_slices = (
    np.rot90(source_volume[:, :, center[2]]),
    np.rot90(source_volume[:, center[1], :]),
    np.rot90(source_volume[center[0], :, :]),
)
mask_slices = (
    np.rot90(brain_mask[:, :, center[2]]),
    np.rot90(brain_mask[:, center[1], :]),
    np.rot90(brain_mask[center[0], :, :]),
)
zscore_slices = (
    np.rot90(normalized_volume[:, :, center[2]]),
    np.rot90(normalized_volume[:, center[1], :]),
    np.rot90(normalized_volume[center[0], :, :]),
)
view_names = ("Axial", "Coronal", "Sagittal")

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(10, 14))
raw_vmin, raw_vmax = np.percentile(brain_values, (1, 99))
for row, (view_name, raw_slice, mask_slice, zscore_slice) in enumerate(
    zip(view_names, raw_slices, mask_slices, zscore_slices)
):
    raw_display = np.ma.masked_where(~mask_slice, raw_slice)
    zscore_display = np.ma.masked_where(~mask_slice, zscore_slice)
    axes[row, 0].imshow(raw_display, cmap="gray", origin="lower", vmin=raw_vmin, vmax=raw_vmax)
    axes[row, 0].set_title(f"FreeSurfer T1.mgz · {view_name}")
    axes[row, 1].imshow(zscore_display, cmap="gray", origin="lower", vmin=-3, vmax=3)
    axes[row, 1].set_title(f"Brain-mask z-score · {view_name}")
    for axis in axes[row]:
        axis.axis("off")

fig.suptitle(f"T1 Intensity Normalization · {sample['participant_id']} · {label}", fontsize=15, y=0.995)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 4))
axes[0].hist(brain_values, bins=80, density=True, color="#277da1", alpha=0.85)
axes[0].set_title("FreeSurfer T1.mgz · Brain-mask intensities")
axes[0].set_xlabel("Intensity (arbitrary units)")
axes[0].set_ylabel("Density")

axes[1].hist(normalized_values, bins=80, density=True, color="#f9844a", alpha=0.85)
axes[1].set_title("Brain-mask z-score · Same voxels")
axes[1].set_xlabel("Standardized intensity (z-score)")
axes[1].set_ylabel("Density")

fig.suptitle("Voxel Intensity Distributions Before and After Per-Volume Z-Score", fontsize=14)
fig.tight_layout()
plt.show()